# 04 - Run the test set and save the results

Runs the 40-question test set through a variant and writes one row per question to
`workspace.policy_rag.experiment_runs`. Same retrieval, prompt and model call as
`03_variant_a`; this adds the persistence that later evaluation -> reads back.

In [0]:
%pip install -q databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
# restartPython() resets sys.path
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repo root:", repo_root)

In [0]:
from src.rag import (RagConfig, RagPipeline, load_questions, new_run_id,
                     retrieval_metrics, run_experiment, save_runs)

cfg = RagConfig(variant="A")
for key, value in cfg.as_dict().items():
    print(f"{key:<20} {value}")
print(f"{'runs_table':<20} {cfg.runs_table}")

## Pick the split

**Don't run `final` until later, as scheduled.** Reading those answers while tuning Variant B is
what stops them and it can't be undone.

In [0]:
SPLIT = "dev" # "dev" | "final" | None for all 40
RUN_LABEL = "variant_a_dev_01"

questions = load_questions(repo_root / "data" / "questions" / "questions.jsonl", split=SPLIT)

from collections import Counter
print(f"{len(questions)} questions ({SPLIT})")
for qtype, n in sorted(Counter(q["question_type"] for q in questions).items()):
    print(f"  {qtype}: {n}")

In [0]:
pipeline = RagPipeline(cfg)
run_id = new_run_id()


def progress(position, total, row):
    print(f"[{position}/{total}] {row['question_id']:<10} {row['error'] or 'ok'}")


rows = run_experiment(pipeline, questions, run_label=RUN_LABEL, run_id=run_id,
                      on_progress=progress)

n_errors = sum(1 for r in rows if r["error"])
print(f"\nrun_id: {run_id}  |  {len(rows)} rows, {n_errors} errors")

In [0]:
table = save_runs(spark, cfg, rows)
print("Saved to", table)

## Retrieval, per question type

The only metric available without a judge, and the one that localises the failure:
a wrong answer where `recall_at_k` < 1 is a retrieval problem, not a generation problem. 
Out-of-scope rows have no gold chunks and are left out.

In [0]:
import statistics

scored = [(r, retrieval_metrics(r)) for r in rows if not r["error"]]

by_type = {}
for row, m in scored:
    if m["recall_at_k"] is not None:
        by_type.setdefault(row["question_type"], []).append(m)

print(f"{'question_type':<24} {'n':>3}  {'recall@k':>9}  {'MRR':>6}  {'full':>7}")
for qtype, ms in sorted(by_type.items()):
    recall = statistics.mean(m["recall_at_k"] for m in ms)
    mrr = statistics.mean(m["reciprocal_rank"] for m in ms)
    full = sum(1 for m in ms if m["recall_at_k"] == 1.0)
    print(f"{qtype:<24} {len(ms):>3}  {recall:>9.2f}  {mrr:>6.2f}  {full:>3}/{len(ms)}")

misses = [(r, m) for r, m in scored
          if m["recall_at_k"] is not None and m["recall_at_k"] < 1.0]
print(f"\nIncomplete retrieval: {len(misses)}")
for row, m in misses:
    missing = sorted(set(row["gold_chunk_ids"]) - set(row["retrieved_chunk_ids"]))
    print(f"  {row['question_id']:<10} recall={m['recall_at_k']:.2f}  missing: {missing}")

## Read the answers

For each near_miss and out_of_scope question, note two things: did it decline, and if not, 
did it invent a rule? The near_miss ones are the point, let's say, if Variant A
declines those too, the refusal intervention has no headroom in RQ2 and that belongs in the report now.

In [0]:
for row in rows:
    if row["error"]:
        continue
    m = retrieval_metrics(row)
    recall = "-" if m["recall_at_k"] is None else f"{m['recall_at_k']:.2f}"
    print("=" * 100)
    print(f"{row['question_id']}  [{row['question_type']}]  "
          f"in_scope={row['in_scope']}  recall@k={recall}")
    print(f"Q: {row['question']}")
    print(f"Retrieved: {row['retrieved_chunk_ids']}")
    print(f"Expected:  {row['gold_answer'] or 'decline and refer onward'}")
    print(f"\nA: {row['answer']}\n")

## Optional: log the run to MLflow

This coming week's metrics attach to the same run. The Delta table stays the source of truth.

In [0]:
import mlflow

with mlflow.start_run(run_name=f"{cfg.variant}_{RUN_LABEL}") as mlflow_run:
    mlflow.log_params(cfg.as_dict())
    mlflow.log_params({"run_id": run_id, "run_label": RUN_LABEL, "split": SPLIT})
    mlflow.log_metrics({"n_questions": len(rows), "n_errors": n_errors})
    mlflow.set_tag("results_table", table)
    print("MLflow run:", mlflow_run.info.run_id)